In [24]:
import os
import glob
import pandas as pd
import numpy as np
import cv2
import matplotlib
%matplotlib inline
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
from tqdm.notebook import tqdm_notebook
import re
tqdm_notebook.pandas()
import tensorflow as tf
from tensorflow.keras.models import load_model
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Activation
from tensorflow.keras.layers import Dropout
from tensorflow.keras.layers import Flatten
from tensorflow.keras.layers import GlobalAveragePooling2D
from tensorflow.keras.layers import Conv2D
from tensorflow.keras.layers import MaxPooling2D
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping
from tensorflow.keras.applications.vgg16 import VGG16
from tensorflow.keras.applications.efficientnet import EfficientNetB7
from tensorflow.keras.applications.vgg16 import preprocess_input
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.preprocessing import image
from keras.utils import np_utils
from sklearn.model_selection import train_test_split
from sklearn import metrics
from sklearn.metrics import f1_score , recall_score, accuracy_score, precision_score, confusion_matrix
import itertools
import pickle
from joblib import dump, load
from tensorflow.keras.optimizers import RMSprop
from keras.layers import Dense, GlobalAveragePooling2D
from keras import layers
from keras.optimizers import Adam
from keras.applications import InceptionV3


In [2]:
Y_train = pd.read_csv('C:\\Users\\yahya\\OneDrive\\Documents\\Rakuten\\save fichier rakuten\\Y_train_final.csv')

Y_train= Y_train.drop("Unnamed: 0", axis=1)

Y_train.head()

,cat_code
0,0
1,18
2,2
3,7
4,25


In [3]:
Y_train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 84916 entries, 0 to 84915
Data columns (total 1 columns):
 #   Column    Non-Null Count  Dtype
---  ------    --------------  -----
 0   cat_code  84916 non-null  int64
dtypes: int64(1)
memory usage: 663.5 KB


In [4]:
Y_train["cat_code"] = Y_train["cat_code"].astype('str')


In [5]:
# Définir la largeur d'affichage des colonnes
pd.set_option('display.max_colwidth', 100)  # Modifier la valeur selon vos besoins
df_train = pd.read_csv('C:\\Users\\yahya\\OneDrive\\Documents\\Rakuten\\save fichier rakuten\\df_train_cleaned.csv')

df_train= df_train.drop("Unnamed: 0", axis=1)

df_train.head()

,designation,description,productid,imageid,nom_img,Texte,prdtypecode,Catégories
0,Olivia: Personalisiertes Notizbuch / 150 Seiten / Punktraster / Ca Din A5 / Rosen-Design,NaN,3804725264,1263597046,image_1263597046_product_3804725264,olivia personalisiertes notizbuch seiten punktraster din rosen design,10,Livres occasions
1,Journal Des Arts (Le) N° 133 Du 28/09/2001 - L'art Et Son Marche Salon D'art Asiatique A Paris -...,NaN,436067568,1008141237,image_1008141237_product_436067568,journal arts art marche salon art asiatique paris jacques barrere francois perrier reforme vente...,2280,"Journaux, revues d'occasion"
2,Grand Stylet Ergonomique Bleu Gamepad Nintendo Wii U - Speedlink Pilot Style,PILOT STYLE Touch Pen de marque Speedlink est 1 stylet ergonomique pour GamePad Nintendo Wii U.<...,201115110,938777978,image_938777978_product_201115110,grand stylet ergonomique bleu gamepad nintendo wii speedlink pilot style pilot style touch pen m...,50,accessories gaming
3,Peluche Donald - Europe - Disneyland 2000 (Marionnette À Doigt),NaN,50418756,457047496,image_457047496_product_50418756,peluche donald europe disneyland marionnette doigt,1280,Jouets petite enfance
4,La Guerre Des Tuques,Luc a des id&eacute;es de grandeur. Il veut organiser un jeu de guerre de boules de neige et s'a...,278535884,1077757786,image_1077757786_product_278535884,guerre tuques luc grandeur veut organiser jeu guerre boules neige arranger vainqueur incontest s...,2705,Livres neufs


# Utlisez ce code pour redimensionner les images à 250
# Pour changer cette format au format code il faut appuyer à la fois sur ESC+Y

In [6]:
# L'emplacement des images resized
emplacement = 'C:\\Users\\yahya\\OneDrive\Documents\\Rakuten\\images\\images\\image_resized\\'

# Fonction pour ajouter l'emplacement au nom de l'image
def ajouter_emplacement(nom_img):
    return emplacement + nom_img + ".jpg"

# Appliquer la fonction à la colonne 'nom_img'
df_train['nom_img'] = df_train['nom_img'].apply(ajouter_emplacement)

df_train.head()

,designation,description,productid,imageid,nom_img,Texte,prdtypecode,Catégories
0,Olivia: Personalisiertes Notizbuch / 150 Seiten / Punktraster / Ca Din A5 / Rosen-Design,NaN,3804725264,1263597046,C:\Users\yahya\OneDrive\Documents\Rakuten\images\images\image_resized\image_1263597046_product_3...,olivia personalisiertes notizbuch seiten punktraster din rosen design,10,Livres occasions
1,Journal Des Arts (Le) N° 133 Du 28/09/2001 - L'art Et Son Marche Salon D'art Asiatique A Paris -...,NaN,436067568,1008141237,C:\Users\yahya\OneDrive\Documents\Rakuten\images\images\image_resized\image_1008141237_product_4...,journal arts art marche salon art asiatique paris jacques barrere francois perrier reforme vente...,2280,"Journaux, revues d'occasion"
2,Grand Stylet Ergonomique Bleu Gamepad Nintendo Wii U - Speedlink Pilot Style,PILOT STYLE Touch Pen de marque Speedlink est 1 stylet ergonomique pour GamePad Nintendo Wii U.<...,201115110,938777978,C:\Users\yahya\OneDrive\Documents\Rakuten\images\images\image_resized\image_938777978_product_20...,grand stylet ergonomique bleu gamepad nintendo wii speedlink pilot style pilot style touch pen m...,50,accessories gaming
3,Peluche Donald - Europe - Disneyland 2000 (Marionnette À Doigt),NaN,50418756,457047496,C:\Users\yahya\OneDrive\Documents\Rakuten\images\images\image_resized\image_457047496_product_50...,peluche donald europe disneyland marionnette doigt,1280,Jouets petite enfance
4,La Guerre Des Tuques,Luc a des id&eacute;es de grandeur. Il veut organiser un jeu de guerre de boules de neige et s'a...,278535884,1077757786,C:\Users\yahya\OneDrive\Documents\Rakuten\images\images\image_resized\image_1077757786_product_2...,guerre tuques luc grandeur veut organiser jeu guerre boules neige arranger vainqueur incontest s...,2705,Livres neufs


In [7]:
df_train["prdtypecode"] = df_train["prdtypecode"].astype('str')

In [8]:
# Ajuster les options d'affichage des colonnes
pd.set_option('display.max_colwidth', None)

# Afficher le contenu complet de la colonne 'nom_img' 
print(df_train['nom_img'].head())


0    C:\Users\yahya\OneDrive\Documents\Rakuten\images\images\image_resized\image_1263597046_product_3804725264.jpg
1     C:\Users\yahya\OneDrive\Documents\Rakuten\images\images\image_resized\image_1008141237_product_436067568.jpg
2      C:\Users\yahya\OneDrive\Documents\Rakuten\images\images\image_resized\image_938777978_product_201115110.jpg
3       C:\Users\yahya\OneDrive\Documents\Rakuten\images\images\image_resized\image_457047496_product_50418756.jpg
4     C:\Users\yahya\OneDrive\Documents\Rakuten\images\images\image_resized\image_1077757786_product_278535884.jpg
Name: nom_img, dtype: object


In [9]:
import cv2

# Charger l'image
image = cv2.imread('C:\\Users\\yahya\\OneDrive\\Documents\\Rakuten\\images\\images\\image_resized\\image_1263597046_product_3804725264.jpg')

# Obtenir les dimensions de l'image
hauteur, largeur = image.shape[:2]

print("Largeur de l'image :", largeur)
print("Hauteur de l'image :", hauteur)


Largeur de l'image : 250
Hauteur de l'image : 250


In [10]:
df_train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 84916 entries, 0 to 84915
Data columns (total 8 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   designation  84916 non-null  object
 1   description  55116 non-null  object
 2   productid    84916 non-null  int64 
 3   imageid      84916 non-null  int64 
 4   nom_img      84916 non-null  object
 5   Texte        84906 non-null  object
 6   prdtypecode  84916 non-null  object
 7   Catégories   84916 non-null  object
dtypes: int64(2), object(6)
memory usage: 5.2+ MB


In [51]:
# Sauvegarder la data 
df_train.to_csv('C:\\Users\\yahya\\OneDrive\\Documents\\Rakuten\\save fichier rakuten\\df_train_image.csv')

# inceptionV3

In [11]:
# importer df_train_img
df_train_img = pd.read_csv('C:\\Users\\yahya\\OneDrive\\Documents\\Rakuten\\save fichier rakuten\\df_train_image.csv')

df_train_img= df_train_img.drop("Unnamed: 0", axis=1)
df_train_img.head(1)

,designation,description,productid,imageid,nom_img,Texte,prdtypecode,Catégories
0,Olivia: Personalisiertes Notizbuch / 150 Seiten / Punktraster / Ca Din A5 / Rosen-Design,NaN,3804725264,1263597046,C:\Users\yahya\OneDrive\Documents\Rakuten\images\images\image_resized\image_1263597046_product_3804725264.jpg,olivia personalisiertes notizbuch seiten punktraster din rosen design,10,Livres occasions


In [12]:
df_train_img["prdtypecode"] = df_train_img["prdtypecode"].astype('str')

In [13]:
df_train_img.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 84916 entries, 0 to 84915
Data columns (total 8 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   designation  84916 non-null  object
 1   description  55116 non-null  object
 2   productid    84916 non-null  int64 
 3   imageid      84916 non-null  int64 
 4   nom_img      84916 non-null  object
 5   Texte        84906 non-null  object
 6   prdtypecode  84916 non-null  object
 7   Catégories   84916 non-null  object
dtypes: int64(2), object(6)
memory usage: 5.2+ MB


In [14]:
df_train_img.shape

(84916, 8)

In [15]:
X_train, X_test, y_train, y_test = train_test_split(df_train_img, Y_train, test_size=0.2,  random_state = 123)
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size = 0.1, random_state = 123)

# Data generateur

In [16]:
img_rows = 250
img_cols = 250
batch_size = 64
epochs = 25
emplacement = 'C:\\Users\\yahya\\OneDrive\\Documents\\Rakuten\\images\\images\\image_resized'

In [17]:
#Générateur de données

train_data_generator = ImageDataGenerator(rescale = 1./255, 
        preprocessing_function = preprocess_input,  # data augmentation
        rotation_range = 10,
        width_shift_range = 0.1,
        height_shift_range = 0.1,
        zoom_range = 1.1,
        horizontal_flip = True
        )

val_data_generator = ImageDataGenerator(rescale = 1./255, 
        preprocessing_function = preprocess_input,  # data augmentation
        rotation_range = 10,
        width_shift_range = 0.1,
        height_shift_range = 0.1,
        zoom_range = 1.1,
        horizontal_flip = True
        )

test_data_generator = ImageDataGenerator(rescale = 1./255, 
    preprocessing_function = preprocess_input)

In [18]:
#Itérateur 
train_generator = train_data_generator.flow_from_dataframe(dataframe=X_train,
                                                          directory=emplacement,
                                                           x_col = "nom_img",
                                                           y_col = "prdtypecode",
                                                           class_mode ="sparse",
                                                           target_size = (250, 250), 
                                                          batch_size = batch_size)

test_generator = test_data_generator.flow_from_dataframe(dataframe=X_test,
                                                          directory=emplacement,
                                                           x_col = "nom_img",
                                                           y_col = "prdtypecode",
                                                           class_mode ="sparse",
                                                           target_size = (250, 250), 
                                                          batch_size = batch_size)
                                   
val_generator = val_data_generator.flow_from_dataframe(dataframe=X_val,
                                                          directory=emplacement,
                                                           x_col = "nom_img",
                                                           y_col = "prdtypecode",
                                                           class_mode ="sparse",
                                                           target_size = (250, 250), 
                                                          batch_size = batch_size)


Found 61138 validated image filenames belonging to 27 classes.
Found 16984 validated image filenames belonging to 27 classes.
Found 6794 validated image filenames belonging to 27 classes.


In [19]:
conda install -c anaconda libhdf5


Retrieving notices: ...working... done
Solving environment: ...working... failed with initial frozen solve. Retrying with flexible solve.
Solving environment: ...working... failed with initial frozen solve. Retrying with flexible solve.

Note: you may need to restart the kernel to use updated packages.



PackagesNotFoundError: The following packages are not available from current channels:

  - libhdf5

Current channels:

  - https://conda.anaconda.org/anaconda/win-64
  - https://conda.anaconda.org/anaconda/noarch
  - https://repo.anaconda.com/pkgs/main/win-64
  - https://repo.anaconda.com/pkgs/main/noarch
  - https://repo.anaconda.com/pkgs/r/win-64
  - https://repo.anaconda.com/pkgs/r/noarch
  - https://repo.anaconda.com/pkgs/msys2/win-64
  - https://repo.anaconda.com/pkgs/msys2/noarch

To search for alternate channels that may provide the conda package you're
looking for, navigate to

    https://anaconda.org

and use the search bar at the top of the page.




In [26]:

%%time
# Chargement du modèle InceptionV3
inceptionV3  = InceptionV3(weights=None, include_top = False, input_shape=(img_rows,img_rows,3))

# Bloquage du blackbone
for layer in inceptionV3.layers:
    layer.trainable = False
    
model = Sequential()
model.add(inceptionV3)
# Ajout des couche de classification
model.add(GlobalAveragePooling2D()) 
model.add(Dense(units = 1024,activation='relu'))

model.add(Dropout(rate = 0.2))
model.add(Dense(units = 512, activation='relu'))

model.add(Dropout(rate = 0.2))
#model.add(Dense(units = 27, activation='softmax'))
model.add(Dense(units = 27, activation='sigmoid'))
model.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 inception_v3 (Functional)   (None, 6, 6, 2048)        21802784  
                                                                 
 global_average_pooling2d (  (None, 2048)              0         
 GlobalAveragePooling2D)                                         
                                                                 
 dense (Dense)               (None, 1024)              2098176   
                                                                 
 dropout (Dropout)           (None, 1024)              0         
                                                                 
 dense_1 (Dense)             (None, 512)               524800    
                                                                 
 dropout_1 (Dropout)         (None, 512)               0         
                                                        

In [27]:
# Compilation
model.compile(loss='sparse_categorical_crossentropy',  # fonction de perte
              optimizer='adam',                 # algorithme de descente de gradient
              metrics=['accuracy'])             # métrique d'évaluation

# Callback

In [28]:
%%time

#Sauvegarder les meilleurs poids du modèle au cours de l'entraînement :
checkpoint = ModelCheckpoint(filepath = 'C:\\Users\\yahya\\OneDrive\\Documents\\Rakuten\\DL\\Tensorflow\\'  , 
                             monitor ='val_accuracy',
                             verbose = 1,
                             save_best_only = True, # (pour que le meilleur modèle ne soit pas écrasé)
                             save_weights_only = False,
                             mode ='max', #(permet de préciser si la métrique doit croître ou décroitre : ici on choisit 'min'
                                           #car la métrique est une perte à minimiser)
                             save_freq='epoch')
#Arrêter l'entraînement si le modèle n'évolue plus (très pratique pour ne pas gérer le
#nombre d'epoch) :
early = EarlyStopping(monitor='val_loss',
                      min_delta = 0,
                      patience = 6, #(nombre d'epochs à attendre avant d'arrêter l'entraînement
                      restore_best_weights = True,
                      verbose = 1,
                      mode = 'min')

CPU times: total: 0 ns
Wall time: 0 ns


In [29]:
step_size_Train = train_generator.n//train_generator.batch_size
step_size_Valid = val_generator.n//val_generator.batch_size
print('step_size_Train : ' , step_size_Train)
print('step_size_Valid  : ' , step_size_Valid)

step_size_Train :  955
step_size_Valid  :  106


In [30]:
%%time
# Fit Train generator
history_incv3 = model.fit(train_generator,                    
                    epochs = epochs,                   
                    steps_per_epoch = step_size_Train,
                    validation_data = val_generator,
                    validation_steps = step_size_Valid,
                    callbacks=[checkpoint,early]
                    
                    )

# j'ai dû arrêter le fit parce que j'ai vu que ça apris beaucoup du temps ( 5 heures pour 4 epochs),\
#et que la valeurs de l'accuracy ne se developpe pas

Epoch 1/25


955/955 [==============================] - ETA: 0s - loss: 3.0988 - accuracy: 0.1203
Epoch 1: val_accuracy improved from -inf to 0.11733, saving model to C:\Users\yahya\OneDrive\Documents\Rakuten\DL\Tensorflow\
INFO:tensorflow:Assets written to: C:\Users\yahya\OneDrive\Documents\Rakuten\DL\Tensorflow\assets


INFO:tensorflow:Assets written to: C:\Users\yahya\OneDrive\Documents\Rakuten\DL\Tensorflow\assets


955/955 [==============================] - 3350s 3s/step - loss: 3.0988 - accuracy: 0.1203 - val_loss: 3.0855 - val_accuracy: 0.1173
Epoch 2/25
955/955 [==============================] - ETA: 0s - loss: 3.0501 - accuracy: 0.1248
Epoch 2: val_accuracy improved from 0.11733 to 0.12751, saving model to C:\Users\yahya\OneDrive\Documents\Rakuten\DL\Tensorflow\
INFO:tensorflow:Assets written to: C:\Users\yahya\OneDrive\Documents\Rakuten\DL\Tensorflow\assets


INFO:tensorflow:Assets written to: C:\Users\yahya\OneDrive\Documents\Rakuten\DL\Tensorflow\assets


955/955 [==============================] - 6629s 7s/step - loss: 3.0501 - accuracy: 0.1248 - val_loss: 3.0209 - val_accuracy: 0.1275
Epoch 3/25
955/955 [==============================] - ETA: 0s - loss: 3.0005 - accuracy: 0.1361
Epoch 3: val_accuracy improved from 0.12751 to 0.13340, saving model to C:\Users\yahya\OneDrive\Documents\Rakuten\DL\Tensorflow\
INFO:tensorflow:Assets written to: C:\Users\yahya\OneDrive\Documents\Rakuten\DL\Tensorflow\assets


INFO:tensorflow:Assets written to: C:\Users\yahya\OneDrive\Documents\Rakuten\DL\Tensorflow\assets


... (19 outputs left out)

INFO:tensorflow:Assets written to: C:\Users\yahya\OneDrive\Documents\Rakuten\DL\Tensorflow\assets


955/955 [==============================] - 9012s 9s/step - loss: 2.8913 - accuracy: 0.1691 - val_loss: 2.8927 - val_accuracy: 0.1716
Epoch 24/25
955/955 [==============================] - ETA: 0s - loss: 2.8891 - accuracy: 0.1712
Epoch 24: val_accuracy improved from 0.17158 to 0.17438, saving model to C:\Users\yahya\OneDrive\Documents\Rakuten\DL\Tensorflow\
INFO:tensorflow:Assets written to: C:\Users\yahya\OneDrive\Documents\Rakuten\DL\Tensorflow\assets


INFO:tensorflow:Assets written to: C:\Users\yahya\OneDrive\Documents\Rakuten\DL\Tensorflow\assets


955/955 [==============================] - 8959s 9s/step - loss: 2.8891 - accuracy: 0.1712 - val_loss: 2.8844 - val_accuracy: 0.1744
Epoch 25/25
955/955 [==============================] - ETA: 0s - loss: 2.8873 - accuracy: 0.1698
Epoch 25: val_accuracy did not improve from 0.17438
955/955 [==============================] - 4518s 5s/step - loss: 2.8873 - accuracy: 0.1698 - val_loss: 2.8874 - val_accuracy: 0.1726
CPU times: total: 4d 4h 47min 57s
Wall time: 1d 4h 35min 3s
